In [8]:
import pandas as pd
import numpy as np
import psycopg

In [9]:
conn = psycopg.connect(
    "dbname=dailyedge_development"
)

In [10]:
START_DATE = "2025-09-01"
END_DATE = "2026-07-07"

RTH_START = "08:30"
RTH_END = "15:15"

TRAIL_DISTANCE = 35
ADD_LEVEL_1 = 35
ADD_LEVEL_2 = 50
TARGET_DISTANCE = 105

In [11]:
query = """
SELECT timestamp, open, high, low, close, volume
FROM CANDLES
WHERE timestamp::date BETWEEN %s AND %s
  AND timestamp::time BETWEEN %s AND %s
ORDER BY timestamp
"""

df = pd.read_sql(
    query,
    conn,
    params=(START_DATE, END_DATE, RTH_START, RTH_END)
)

df.head()

/tmp/ipykernel_100256/250686337.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,timestamp,open,high,low,close,volume
0,2025-09-01 08:30:00,24389.135089,24390.693434,24372.512744,24372.772468,539
1,2025-09-01 08:31:00,24373.811365,24377.447503,24353.552883,24356.929296,679
2,2025-09-01 08:32:00,24357.189020,24367.318262,24357.189020,24358.487641,389
3,2025-09-01 08:33:00,24359.266813,24369.136331,24359.266813,24366.279365,420
4,2025-09-01 08:34:00,24367.318262,24375.889158,24366.539089,24371.733572,359


In [14]:
def get_first_to_target(session, target_distance=105):
    entry = session.iloc[0]["open"]
    upper_target = entry + target_distance
    lower_target = entry - target_distance

    for _, candle in session.iterrows():
        hit_long = candle["high"] >= upper_target
        hit_short = candle["low"] <= lower_target

        if hit_long and hit_short:
            return "Ambiguous"

        if hit_long:
            return "Long"

        if hit_short:
            return "Short"

    return "Neither"

In [15]:
session_directions = (
    df.groupby(df["timestamp"].dt.date)
      .apply(get_first_to_target)
)

session_directions.value_counts()

Long       109
Short       96
Neither     14
Name: count, dtype: int64

In [18]:
def evaluate_pyramid_baseline(session, direction):
    entry = session.iloc[0]["open"]

    if direction == "Long":
        add_1 = entry + ADD_LEVEL_1
        add_2 = entry + ADD_LEVEL_2
        target = entry + TARGET_DISTANCE
        stop = entry - TRAIL_DISTANCE
        favorable_extreme = entry

    elif direction == "Short":
        add_1 = entry - ADD_LEVEL_1
        add_2 = entry - ADD_LEVEL_2
        target = entry - TARGET_DISTANCE
        stop = entry + TRAIL_DISTANCE
        favorable_extreme = entry

    else:
        return None

    unit_entries = [entry]
    added_1 = False
    added_2 = False

    if direction == "Long":
        for _, candle in session.iterrows():
            # Stop entering this candle is fixed.
            if candle["open"] <= stop:
                exit_price = candle["open"]
                return sum(exit_price - unit_entry for unit_entry in unit_entries)

            hit_stop = candle["low"] <= stop
            hit_target = candle["high"] >= target

            if hit_stop and hit_target:
                return np.nan

            # Any add level below a reached target was necessarily crossed first.
            if hit_target:
                if not added_1:
                    unit_entries.append(add_1)
                if not added_2:
                    unit_entries.append(add_2)

                return sum(target - unit_entry for unit_entry in unit_entries)

            if hit_stop:
                return sum(stop - unit_entry for unit_entry in unit_entries)

            if not added_1 and candle["high"] >= add_1:
                unit_entries.append(add_1)
                added_1 = True

            if not added_2 and candle["high"] >= add_2:
                unit_entries.append(add_2)
                added_2 = True

            favorable_extreme = max(favorable_extreme, candle["high"])
            stop = max(stop, favorable_extreme - TRAIL_DISTANCE)

In [23]:
def evaluate_pyramid_baseline(session, direction):
    entry = session.iloc[0]["open"]

    if direction == "Long":
        add_1 = entry + ADD_LEVEL_1
        add_2 = entry + ADD_LEVEL_2
        target = entry + TARGET_DISTANCE
        stop = entry - TRAIL_DISTANCE
        favorable_extreme = entry

    elif direction == "Short":
        add_1 = entry - ADD_LEVEL_1
        add_2 = entry - ADD_LEVEL_2
        target = entry - TARGET_DISTANCE
        stop = entry + TRAIL_DISTANCE
        favorable_extreme = entry

    else:
        return None

    unit_entries = [entry]
    added_1 = False
    added_2 = False

    if direction == "Long":
        for _, candle in session.iterrows():
            if candle["open"] <= stop:
                exit_price = candle["open"]
                return sum(exit_price - unit_entry for unit_entry in unit_entries)

            hit_stop = candle["low"] <= stop
            hit_target = candle["high"] >= target

            if hit_stop and hit_target:
                return np.nan

            if hit_target:
                if not added_1:
                    unit_entries.append(add_1)
                if not added_2:
                    unit_entries.append(add_2)

                return sum(target - unit_entry for unit_entry in unit_entries)

            if hit_stop:
                return sum(stop - unit_entry for unit_entry in unit_entries)

            if not added_1 and candle["high"] >= add_1:
                unit_entries.append(add_1)
                added_1 = True

            if not added_2 and candle["high"] >= add_2:
                unit_entries.append(add_2)
                added_2 = True

            favorable_extreme = max(favorable_extreme, candle["high"])
            stop = max(stop, favorable_extreme - TRAIL_DISTANCE)

    else:
        for _, candle in session.iterrows():
            if candle["open"] >= stop:
                exit_price = candle["open"]
                return sum(unit_entry - exit_price for unit_entry in unit_entries)

            hit_stop = candle["high"] >= stop
            hit_target = candle["low"] <= target

            if hit_stop and hit_target:
                return np.nan

            if hit_target:
                if not added_1:
                    unit_entries.append(add_1)
                if not added_2:
                    unit_entries.append(add_2)

                return sum(unit_entry - target for unit_entry in unit_entries)

            if hit_stop:
                return sum(unit_entry - stop for unit_entry in unit_entries)

            if not added_1 and candle["low"] <= add_1:
                unit_entries.append(add_1)
                added_1 = True

            if not added_2 and candle["low"] <= add_2:
                unit_entries.append(add_2)
                added_2 = True

            favorable_extreme = min(favorable_extreme, candle["low"])
            stop = min(stop, favorable_extreme + TRAIL_DISTANCE)

    exit_price = session.iloc[-1]["close"]

    if direction == "Long":
        return sum(exit_price - unit_entry for unit_entry in unit_entries)

    return sum(unit_entry - exit_price for unit_entry in unit_entries)

In [24]:
baseline_results = []

for date, direction in session_directions.items():
    if direction not in ("Long", "Short"):
        continue

    session = df[df["timestamp"].dt.date == date]

    pnl = evaluate_pyramid_baseline(session, direction)

    baseline_results.append({
        "Date": date,
        "Direction": direction,
        "PnL": pnl
    })

baseline_results = pd.DataFrame(baseline_results)

baseline_results["PnL"].describe()

count    204.000000
mean      23.698775
std       84.332533
min      -97.750000
25%      -35.000000
50%      -13.923566
75%       42.618239
max      230.000000
Name: PnL, dtype: float64

In [25]:
resolved_pnl = baseline_results["PnL"].dropna()

print("Resolved trades:", len(resolved_pnl))
print("Average P&L:", resolved_pnl.mean())
print("Median P&L:", resolved_pnl.median())
print("Winning trades:", (resolved_pnl > 0).sum())
print("Losing trades:", (resolved_pnl < 0).sum())
print("Breakeven trades:", (resolved_pnl == 0).sum())
print("Average winner:", resolved_pnl[resolved_pnl > 0].mean())
print("Average loser:", resolved_pnl[resolved_pnl < 0].mean())

Resolved trades: 204
Average P&L: 23.69877450980383
Median P&L: -13.923565500001132
Winning trades: 68
Losing trades: 136
Breakeven trades: 0
Average winner: 119.02290623529386
Average loser: -23.963291352941187


In [26]:
wrong_results = []

for date, direction in session_directions.items():
    if direction not in ("Long", "Short"):
        continue

    wrong_direction = "Short" if direction == "Long" else "Long"
    session = df[df["timestamp"].dt.date == date]

    pnl = evaluate_pyramid_baseline(session, wrong_direction)

    wrong_results.append({
        "Date": date,
        "Direction": wrong_direction,
        "PnL": pnl
    })

wrong_results = pd.DataFrame(wrong_results)

wrong_results["PnL"].describe()

count    205.000000
mean     -21.898390
std       24.227984
min     -131.206324
25%      -35.000000
50%      -29.342104
75%      -13.608346
max       96.495365
Name: PnL, dtype: float64

In [27]:
avg_correct_pnl = baseline_results["PnL"].mean()
avg_wrong_pnl = wrong_results["PnL"].mean()

accuracies = np.arange(0.50, 0.81, 0.05)

expectancy = pd.DataFrame({
    "Accuracy": accuracies,
    "Expectancy": [
        accuracy * avg_correct_pnl
        + (1 - accuracy) * avg_wrong_pnl
        for accuracy in accuracies
    ]
})

expectancy

,Accuracy,Expectancy
0,0.50,0.900192
1,0.55,3.180050
2,0.60,5.459909
3,0.65,7.739767
4,0.70,10.019625
5,0.75,12.299483
6,0.80,14.579342


In [28]:
baseline_summary = {
    "Strategy": "Add 35/50, all out 105",
    "Avg Correct PnL": avg_correct_pnl,
    "Avg Wrong PnL": avg_wrong_pnl,
    "Expectancy 60%": (
        0.60 * avg_correct_pnl
        + 0.40 * avg_wrong_pnl
    ),
    "Avg Losing Trade": resolved_pnl[resolved_pnl < 0].mean(),
}

baseline_summary

{'Strategy': 'Add 35/50, all out 105',
 'Avg Correct PnL': np.float64(23.69877450980383),
 'Avg Wrong PnL': np.float64(-21.89839037073185),
 'Expectancy 60%': np.float64(5.459908557589557),
 'Avg Losing Trade': np.float64(-23.963291352941187)}

In [31]:
def evaluate_pyramid_scaleout(session, direction):
    entry = session.iloc[0]["open"]

    if direction == "Long":
        add_1 = entry + ADD_LEVEL_1
        add_2 = entry + ADD_LEVEL_2
        target_1 = entry + 70
        target_2 = entry + 85
        target_3 = entry + 105
        stop = entry - TRAIL_DISTANCE
        favorable_extreme = entry

    elif direction == "Short":
        add_1 = entry - ADD_LEVEL_1
        add_2 = entry - ADD_LEVEL_2
        target_1 = entry - 70
        target_2 = entry - 85
        target_3 = entry - 105
        stop = entry + TRAIL_DISTANCE
        favorable_extreme = entry

    else:
        return None

    active_units = [entry]
    realized_pnl = 0.0

    added_1 = False
    added_2 = False
    took_70 = False
    took_85 = False

    if direction == "Long":
        for _, candle in session.iterrows():

            # Stop entering this candle is fixed.
            if candle["open"] <= stop:
                exit_price = candle["open"]
                realized_pnl += sum(
                    exit_price - unit_entry
                    for unit_entry in active_units
                )
                return realized_pnl

            hit_stop = candle["low"] <= stop

            next_target = (
                target_1 if not took_70
                else target_2 if not took_85
                else target_3
            )

            hit_next_target = candle["high"] >= next_target

            # Existing stop and next scale-out level both touched:
            # 1-minute OHLC cannot establish ordering.
            if hit_stop and hit_next_target:
                return np.nan

            if hit_stop:
                realized_pnl += sum(
                    stop - unit_entry
                    for unit_entry in active_units
                )
                return realized_pnl

            # Add units as favorable thresholds are crossed.
            if not added_1 and candle["high"] >= add_1:
                active_units.append(add_1)
                added_1 = True

            if not added_2 and candle["high"] >= add_2:
                active_units.append(add_2)
                added_2 = True

            # FIFO scale-outs.
            if not took_70 and candle["high"] >= target_1:
                unit_entry = active_units.pop(0)
                realized_pnl += target_1 - unit_entry
                took_70 = True

            if not took_85 and candle["high"] >= target_2:
                unit_entry = active_units.pop(0)
                realized_pnl += target_2 - unit_entry
                took_85 = True

            if candle["high"] >= target_3:
                unit_entry = active_units.pop(0)
                realized_pnl += target_3 - unit_entry
                return realized_pnl

            # New trailing stop becomes active NEXT candle.
            favorable_extreme = max(
                favorable_extreme,
                candle["high"]
            )

            stop = max(
                stop,
                favorable_extreme - TRAIL_DISTANCE
            )

In [33]:
def evaluate_pyramid_scaleout(session, direction):
    entry = session.iloc[0]["open"]

    if direction == "Long":
        add_1 = entry + ADD_LEVEL_1
        add_2 = entry + ADD_LEVEL_2
        target_1 = entry + 70
        target_2 = entry + 85
        target_3 = entry + 105
        stop = entry - TRAIL_DISTANCE
        favorable_extreme = entry

    elif direction == "Short":
        add_1 = entry - ADD_LEVEL_1
        add_2 = entry - ADD_LEVEL_2
        target_1 = entry - 70
        target_2 = entry - 85
        target_3 = entry - 105
        stop = entry + TRAIL_DISTANCE
        favorable_extreme = entry

    else:
        return None

    active_units = [entry]
    realized_pnl = 0.0

    added_1 = False
    added_2 = False
    took_70 = False
    took_85 = False

    if direction == "Long":
        for _, candle in session.iterrows():

            if candle["open"] <= stop:
                exit_price = candle["open"]
                realized_pnl += sum(
                    exit_price - unit_entry
                    for unit_entry in active_units
                )
                return realized_pnl

            hit_stop = candle["low"] <= stop

            next_target = (
                target_1 if not took_70
                else target_2 if not took_85
                else target_3
            )

            hit_next_target = candle["high"] >= next_target

            if hit_stop and hit_next_target:
                return np.nan

            if hit_stop:
                realized_pnl += sum(
                    stop - unit_entry
                    for unit_entry in active_units
                )
                return realized_pnl

            if not added_1 and candle["high"] >= add_1:
                active_units.append(add_1)
                added_1 = True

            if not added_2 and candle["high"] >= add_2:
                active_units.append(add_2)
                added_2 = True

            if not took_70 and candle["high"] >= target_1:
                unit_entry = active_units.pop(0)
                realized_pnl += target_1 - unit_entry
                took_70 = True

            if not took_85 and candle["high"] >= target_2:
                unit_entry = active_units.pop(0)
                realized_pnl += target_2 - unit_entry
                took_85 = True

            if candle["high"] >= target_3:
                unit_entry = active_units.pop(0)
                realized_pnl += target_3 - unit_entry
                return realized_pnl

            favorable_extreme = max(
                favorable_extreme,
                candle["high"]
            )

            stop = max(
                stop,
                favorable_extreme - TRAIL_DISTANCE
            )

    else:
        for _, candle in session.iterrows():

            if candle["open"] >= stop:
                exit_price = candle["open"]
                realized_pnl += sum(
                    unit_entry - exit_price
                    for unit_entry in active_units
                )
                return realized_pnl

            hit_stop = candle["high"] >= stop

            next_target = (
                target_1 if not took_70
                else target_2 if not took_85
                else target_3
            )

            hit_next_target = candle["low"] <= next_target

            if hit_stop and hit_next_target:
                return np.nan

            if hit_stop:
                realized_pnl += sum(
                    unit_entry - stop
                    for unit_entry in active_units
                )
                return realized_pnl

            if not added_1 and candle["low"] <= add_1:
                active_units.append(add_1)
                added_1 = True

            if not added_2 and candle["low"] <= add_2:
                active_units.append(add_2)
                added_2 = True

            if not took_70 and candle["low"] <= target_1:
                unit_entry = active_units.pop(0)
                realized_pnl += unit_entry - target_1
                took_70 = True

            if not took_85 and candle["low"] <= target_2:
                unit_entry = active_units.pop(0)
                realized_pnl += unit_entry - target_2
                took_85 = True

            if candle["low"] <= target_3:
                unit_entry = active_units.pop(0)
                realized_pnl += unit_entry - target_3
                return realized_pnl

            favorable_extreme = min(
                favorable_extreme,
                candle["low"]
            )

            stop = min(
                stop,
                favorable_extreme + TRAIL_DISTANCE
            )

    # Flatten any remaining units at the final RTH close.
    exit_price = session.iloc[-1]["close"]

    if direction == "Long":
        realized_pnl += sum(
            exit_price - unit_entry
            for unit_entry in active_units
        )
    else:
        realized_pnl += sum(
            unit_entry - exit_price
            for unit_entry in active_units
        )

    return realized_pnl

In [34]:
scaleout_results = []

for date, direction in session_directions.items():
    if direction not in ("Long", "Short"):
        continue

    session = df[df["timestamp"].dt.date == date]

    pnl = evaluate_pyramid_scaleout(session, direction)

    scaleout_results.append({
        "Date": date,
        "Direction": direction,
        "PnL": pnl
    })

scaleout_results = pd.DataFrame(scaleout_results)

scaleout_results["PnL"].describe()

count    199.000000
mean      23.020567
std       75.458479
min      -97.750000
25%      -35.000000
50%      -14.425833
75%       69.318393
max      175.000000
Name: PnL, dtype: float64

In [35]:
scaleout_wrong_results = []

for date, direction in session_directions.items():
    if direction not in ("Long", "Short"):
        continue

    wrong_direction = "Short" if direction == "Long" else "Long"
    session = df[df["timestamp"].dt.date == date]

    pnl = evaluate_pyramid_scaleout(session, wrong_direction)

    scaleout_wrong_results.append({
        "Date": date,
        "Direction": wrong_direction,
        "PnL": pnl
    })

scaleout_wrong_results = pd.DataFrame(scaleout_wrong_results)

scaleout_wrong_results["PnL"].describe()

count    204.000000
mean     -19.754097
std       31.470264
min     -131.206324
25%      -35.000000
50%      -29.366020
75%      -13.608346
max      130.498455
Name: PnL, dtype: float64

In [36]:
avg_scaleout_correct = scaleout_results["PnL"].mean()
avg_scaleout_wrong = scaleout_wrong_results["PnL"].mean()

comparison = pd.DataFrame({
    "Accuracy": accuracies,
    "All Out 105": [
        accuracy * avg_correct_pnl
        + (1 - accuracy) * avg_wrong_pnl
        for accuracy in accuracies
    ],
    "Scale 70/85/105": [
        accuracy * avg_scaleout_correct
        + (1 - accuracy) * avg_scaleout_wrong
        for accuracy in accuracies
    ]
})

comparison["Difference"] = (
    comparison["Scale 70/85/105"]
    - comparison["All Out 105"]
)

comparison

,Accuracy,All Out 105,Scale 70/85/105,Difference
0,0.50,0.900192,1.633235,0.733043
1,0.55,3.180050,3.771968,0.591917
2,0.60,5.459909,5.910701,0.450792
3,0.65,7.739767,8.049434,0.309667
4,0.70,10.019625,10.188167,0.168542
5,0.75,12.299483,12.326901,0.027417
6,0.80,14.579342,14.465634,-0.113708


In [37]:
scaleout_resolved = scaleout_results["PnL"].dropna()

print("Resolved trades:", len(scaleout_resolved))
print("Average P&L:", scaleout_resolved.mean())
print("Median P&L:", scaleout_resolved.median())
print("Winning trades:", (scaleout_resolved > 0).sum())
print("Losing trades:", (scaleout_resolved < 0).sum())
print("Breakeven trades:", (scaleout_resolved == 0).sum())
print("Average winner:", scaleout_resolved[scaleout_resolved > 0].mean())
print("Average loser:", scaleout_resolved[scaleout_resolved < 0].mean())

Resolved trades: 199
Average P&L: 23.020566608040173
Median P&L: -14.425833000001148
Winning trades: 65
Losing trades: 134
Breakeven trades: 0
Average winner: 120.01648713846141
Average loser: -24.02969335074625
